# Interactive map showing AOIs associated with each CSDA Evaluation Site  

*Notes:*  
+ AOIs for CSDA sites (a lat, lon):
  + default to 3 km radius circles
  + can adjusted to be 3 km length boxes
  + can be a custom AOI (provide me with a single custom AOI as a geojson file).

Paul Montesano, PhD  
2026

In [1]:
library(sf)
library(leaflet)
library(dplyr)

Linking to GEOS 3.12.2, GDAL 3.9.1, PROJ 9.4.1; sf_use_s2() is TRUE

Warning message:
“package ‘leaflet’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [4]:
# Read the GeoJSON file
sites <- st_read("../sites/csda_sites_aoi.geojson")

Reading layer `csda_sites_aoi' from data source 
  `/panfs/ccds02/home/pmontesa/code/csda_summaries/sites/csda_sites_aoi.geojson' 
  using driver `GeoJSON'
Simple feature collection with 115 features and 15 fields
Geometry type: POLYGON
Dimension:     XY
Bounding box:  xmin: -123.6845 ymin: -51.63934 xmax: 145.0026 ymax: 67.42133
Geodetic CRS:  WGS 84


In [15]:
colnames(sites)

[1] "Site.Name.abbrev"      "Site.Name"             "Location.Name"        
 [4] "Country"               "Longitude"             "Latitude"             
 [7] "Remote.Sensing.Domain" "Priority.Level"        "Evaluation.Category"  
[10] "Source"                "Surface.Domain"        "Assessment.type.s."   
[13] "aoi_type"              "aoi_size_km"           "area_km2"             
[16] "geometry"

In [24]:
# Create 25 km radius circles to show the vicinity around each CSDA site centroid
sites_buffered <- sites %>%
  st_transform(4326) %>%  # Ensure WGS84
  st_centroid() %>%
  st_transform(3857) %>%  # Transform to metric CRS for buffering
  st_buffer(dist = 25000) %>%  # 25 km = 25,000 meters
  st_transform(4326)  # Transform back to WGS84 for leaflet

# Create color palette based on Evaluation.Category
eval_categories <- unique(sites$Remote.Sensing.Domain)
color_pal <- colorFactor("Set3", domain = eval_categories)

# Create an interactive map
csda_sites_map = leaflet(height=800) %>%
  addProviderTiles(providers$Esri.WorldGrayCanvas, group = "Basemap") %>%
  addProviderTiles(providers$OpenStreetMap, group = "Street Map") %>%
  addProviderTiles(providers$Esri.WorldImagery, group = "Satellite") %>%
  addProviderTiles(providers$Esri.WorldTerrain, group="Terrain") %>%

  # Add 25 km radius circles (drawn first so they're behind)
  addPolygons(
    data = sites_buffered,
    fillColor = "transparent",
    fillOpacity = 0,
    color = "black",
    weight = 1,
    dashArray = "5, 5",  # Dotted line
    opacity = 0.7,
    group = "25 km Buffers"
  ) %>%
  
  # Add actual site polygons
  addPolygons(
    data = sites,
    fillColor = ~color_pal(Remote.Sensing.Domain),
    fillOpacity = 0.7,
    color = "white",
    weight = 2,
    popup = ~paste0(
      "<b>Site Name:</b> ", Site.Name, "<br>",
      "<b>SME Domainy:</b> ", Remote.Sensing.Domain, "<br>",
      "<b>Area (km²):</b> ", round(st_area(geometry) / 1e6, 2)
    ),
    label = ~Site.Name,
    highlightOptions = highlightOptions(
      weight = 3,
      color = "red",
      fillOpacity = 0.9,
      bringToFront = TRUE
    ),
    group = "AOIs of CSDA Sites"
  ) %>%
  
  # Add layers control
  addLayersControl(
    baseGroups = c("Basemap","Satellite", "Street Map","Terrain"),
    overlayGroups = c("AOIs of CSDA sites", "Vicinity of CSDA Sites" ),
    options = layersControlOptions(collapsed = FALSE)
  ) %>%
  
  # Add legend
  addLegend(
    position = "bottomright",
    pal = color_pal,
    values = sites$Remote.Sensing.Domain,
    title = "SME Domain",
    opacity = 0.7
  ) %>%

  addScaleBar(
      position = "bottomleft",  # or "topleft", "topright", "bottomright"
      options = scaleBarOptions(
        maxWidth = 100,         # width in pixels
        metric = TRUE,          # show metric units (km/m)
        imperial = TRUE,        # show imperial units (mi/ft)
        updateWhenIdle = FALSE  # update during zoom/pan
      )
    )

Warning message:
“st_centroid assumes attributes are constant over geometries”


In [ ]:
library(htmlwidgets)

In [26]:
# Save map
saveWidget(csda_sites_map, file = "csda_eval_sites_map.html", selfcontained = TRUE)